In [1]:
#!git clone https://github.com/DataTalksClub/machine-learning-zoomcamp.git

In [2]:
%cd /workspaces/ml_zoomcamp/05-deployment/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment

/workspaces/ml_zoomcamp/05-deployment/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment


In [3]:
import os
os.listdir()

['.python-version',
 'pipeline.bin',
 'README.md',
 'model.py',
 'model_metadata.json',
 '.venv',
 'feature_defaults.json',
 'homework.md',
 'q6_test.py',
 'train.py',
 'pyproject.toml',
 'smoke_test.py',
 'homework.yaml',
 'uv.lock',
 'Dockerfile',
 '__pycache__',
 'predict.py']

In [4]:
%pip install uv

Note: you may need to restart the kernel to use updated packages.


In [5]:
## Question 1
!uv --version

uv 0.13.5 (x86_64-unknown-linux-gnu)


In [6]:
!uv python install 3.11.15
!uv sync --locked


Python 3.11.15 is already installed
Resolved 19 packages in 0.93ms
Checked 18 packages in 9ms


In [7]:
!uv run python smoke_test.py

{'conversion_probability': 0.769799, 'conversion': True}


In [8]:
## Question 2
#scikit-learn==1.7.2

In [9]:
## Question 3

import pickle

with open("pipeline.bin", "rb") as f:
    model = pickle.load(f)

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41
}

probability = model.predict_proba([lead])[0, 1]
print(f"Conversion probability: {probability:.3f}")


/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Conversion probability: 0.533


/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [10]:
import json
import urllib.request

lead = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
}

request = urllib.request.Request(
    "http://localhost:9696/predict",
    data=json.dumps(lead).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with urllib.request.urlopen(request) as response:
    result = json.load(response)

print(result)
probability = result["conversion_probability"]
print(f"{probability:.3f}")


{'conversion_probability': 0.769799, 'conversion': True}
0.770


In [11]:
## Question 5
!cat Dockerfile

!python q6_test.py

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3

COPY --from=ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4 /uv /uvx /bin/

ENV PATH="/code/.venv/bin:$PATH" \
    UV_COMPILE_BYTECODE=1 \
    UV_LINK_MODE=copy

WORKDIR /code
COPY pyproject.toml uv.lock .python-version ./
RUN uv sync --locked --no-dev

COPY model.py predict.py feature_defaults.json model_metadata.json pipeline.bin q6_test.py ./

EXPOSE 9696
ENTRYPOINT ["uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"]
{'conversion_probability': 0.769799, 'conversion': True}


In [13]:
!curl -s http://localhost:9696/health

{"status":"ok","model_sha256":"1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1","feature_defaults_sha256":"1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd"}